# 🏁 Notebook 2: Axis Activation & Coordinate Alignment

This notebook supplies power to the physical axes and aligns the coordinate system, preparing the hardware for precise motion.

### 🎯 Objective & Scope
* **Focus**: Clears hardware alarms, applies power (Servo On), establishes the absolute coordinate origin (Zeroing), and configures the control mode.
* **Prerequisite**: Notebook 1 must be successfully completed to ensure the WMX3 engine and EtherCAT network are active.

### 🔄 State Machine & Initialization Sequence
* **Step 5: Clear Axis Alarms**: Reset any active error latches on the servo drives.
* **Step 6: Motor Servo On**: Inject current to the motors to lock the axes securely in place.
* **Step 7: Execute Axis Homing**: Define the current physical position as the software's absolute origin (0.0).
* **Step 8: Set Control Mode (Position)**: Configure the axes for precise, position-based trajectory control.

In [ ]:
import rclpy
from wmx_r2_message.msg import AxisVelocity
from wmx_r2_message.srv import SetAxis
import ipywidgets as widgets
from IPython.display import display
import threading
import time

from wmx_utils import WmxClient

if not rclpy.ok():
    rclpy.init()

wmx = WmxClient(node_name='wmx_client_02')
print(f"✅ WMX Client Ready for Notebook 02. Target Axes: {wmx.axis_list}")

## Step 5: Clear Axis Alarms

Before applying power, reset any residual hardware alarms (e.g., initial boot errors, emergency stops) on the servo drives to ensure a safe state.

In [ ]:
req = SetAxis.Request()
req.index = wmx.axis_list
req.data = [0] * len(wmx.axis_list)

res = wmx.call(SetAxis, '/wmx/axis/clear_alarm', req)
if res is not None:
    print(f"✅ [SUCCESS]\n{res.message}".replace("; ", "\n").strip())
else:
    print("❌ [ERROR] Failed to request alarm clear.")

## Step 6: Motor Servo On

Apply power to the axes and release the hardware brakes. Once successful, the motors will firmly hold their current positions (Locked).

In [ ]:
req = SetAxis.Request()
req.index = wmx.axis_list
req.data = [0] * len(wmx.axis_list)

res = wmx.call(SetAxis, '/wmx/axis/set_on', req)
if res is not None:
    print(f"{res.message}".replace("; ", "\n").strip())
else:
    print("❌ [ERROR] Failed to enable servo (Servo On).")

## Step 7: Execute Axis Homing

Set the current physical position as the absolute zero point (0.0) in the software coordinate system, establishing a reference for all future movements.

In [ ]:
req = SetAxis.Request()
req.index = wmx.axis_list
req.data = [0] * len(wmx.axis_list)

res = wmx.call(SetAxis, '/wmx/axis/homing', req)
if res is not None:
    msg = res.message.replace('; ', '\n').replace(';', '\n').replace(', Axis', '\nAxis').replace(',Axis', '\nAxis')
    print(f"✅ [SUCCESS] Homing Results (Zero registry updated):\n{msg}")
else:
    print("❌ [ERROR] Failed to complete homing sequence.")

## Step 8: Set Control Mode (Position)

Change the control mode of the axes to **Position Mode**. The hardware is now fully prepared to execute precise positional commands!

In [ ]:
req = SetAxis.Request()
req.index = wmx.axis_list
req.data = [1] * len(wmx.axis_list)

res = wmx.call(SetAxis, '/wmx/axis/set_mode', req)
if res is not None:
    msg = res.message.replace('; ', '\n').replace(';', '\n').replace(', Axis', '\nAxis').replace(',Axis', '\nAxis')
    print(f"🚀 [SUCCESS] Control Mode Results:\n{msg}")
    print("\nAll systems are fully homed and locked in Position Mode! Ready to move.")
else:
    print("❌ [ERROR] Failed to configure control mode.")